# Smooth Scaling Laws Hide Stepwise Token Learning: Quickstart

<a href="https://colab.research.google.com/github/applewpj/stepwise-token-learning/blob/main/notebooks/quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>
<a href="https://openreview.net/forum?id=THZDVZArRH"><img src="https://img.shields.io/badge/NeurIPS%202026-Spotlight-4B44CE" alt="NeurIPS 2026 Spotlight"></a>
<a href="https://github.com/applewpj/stepwise-token-learning"><img src="https://img.shields.io/badge/Code-GitHub-181717?logo=github" alt="Code"></a>
<a href="https://huggingface.co/datasets/applewpj/token-learning-spectrum-examples"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-loss%20matrices-FFD21E" alt="Dataset"></a>

Individual token losses do not decay as power laws: each token is learned in a localized, sigmoid-like step at its own *learning time* $\tau$. The distribution of those times, the **learning-time spectrum** $p(\tau)$, is what shapes the smooth scaling law.

This notebook reproduces the core analysis in a few minutes, without a GPU:

1. **Synthetic warm-up** (about 30 s): the full pipeline on 10,000 synthetic arithmetic samples.
2. **Real pre-training data** (1–2 min on CPU): fit 50,000 tokens from the training-step matrix of a 1.5B model trained on 300B tokens, then reconstruct the loss derivative of all one million tokens from their learning-time spectrum.

A GPU runtime (*Runtime → Change runtime type*) makes Part 2 faster but is not required.

In [ ]:
import os
import sys

if not os.path.isdir("tokenlearning"):
    !git clone -q --depth 1 https://github.com/applewpj/stepwise-token-learning
    %cd stepwise-token-learning
%pip install -q .

## Part 1 · Synthetic warm-up

Each row of this matrix is one arithmetic expression from the synthetic *Mano* task (adapted from [PhysicsLM4](https://github.com/facebookresearch/PhysicsLM4)), with difficulty levels L1–L10, scored at 80 checkpoints of one training run. The pipeline fits a sigmoid learning event to every row, estimates the learning-time spectrum, and reconstructs the loss derivative. `reproduce` downloads the matrix (1.5 MB), runs the paper config, and checks the result against the reference numbers in `configs/reference_metrics.yaml`; from a shell, the same command is `tokenlearning reproduce synthetic`.

In [ ]:
from tokenlearning.reproduce import reproduce

[synthetic] = reproduce(["synthetic"])

In [ ]:
from IPython.display import Image, display

figures = "runs/synthetic_mano_uniform_validation_10k/figures"
display(Image(f"{figures}/token_examples_by_difficulty.png", width=900))
display(Image(f"{figures}/mean_loss_by_difficulty.png", width=620))
display(Image(f"{figures}/rate_reconstruction.png", width=620))

**What to look for.** Individual samples drop in sharp steps (red dashed lines are the sigmoid fits), at times that depend on the sample, while the mean loss of each difficulty level declines smoothly. In the last plot, one shared learning pulse placed at every measured learning time (green, *spectrum-dominated derivative*) nearly coincides with the derivative implied by the per-sample fits (red) and follows the trend of the noisy empirical derivative of the 10,000-sample average (blue).

## Part 2 · Real pre-training data: the training-step axis

`step_axis_1m` holds the validation loss of **1,000,000 contextualized tokens** at 54 checkpoints (steps 15k–71k) of a 1.5B model trained on 300B tokens, the data behind Figure 3 of the paper. Fitting every row for 10,000 epochs, as in the paper, is a GPU job. Here we fit a random 5% sample and ask:

> Does the learning-time spectrum of 50,000 tokens reconstruct the loss derivative of all one million?

The download is 197 MB.

In [ ]:
!{sys.executable} -m tokenlearning download step_axis_1m

import numpy as np
from tokenlearning.io import load_loss_npz

data = load_loss_npz("examples/step_axis_1m/losses.npz")
steps, losses = data.axis_values, data.loss_matrix
print(f"{losses.shape[0]:,} tokens x {losses.shape[1]} checkpoints, steps {steps[0]:,.0f} to {steps[-1]:,.0f}")

Fit a learning event to each sampled token with the paper's settings from `configs/step_axis_1m.yaml`, then apply the paper's filters. On a GPU runtime you can set `N_TOKENS = 1_000_000` to fit the full matrix.

In [ ]:
import torch
import yaml
from tokenlearning.core import fit_token_events, filter_tokens

N_TOKENS = 50_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = yaml.safe_load(open("configs/step_axis_1m.yaml"))
fit_cfg = {
    **cfg["fit"],
    "device": DEVICE,
    "epochs": 10_000 if DEVICE == "cuda" else 2_000,
    "chunk_size": min(N_TOKENS, cfg["fit"]["chunk_size"]),
    "log_every": 0,
}

rows = np.sort(np.random.default_rng(0).choice(losses.shape[0], N_TOKENS, replace=False))
fit = fit_token_events(steps, losses[rows], fit_cfg)
mask = filter_tokens(fit, cfg["filters"])
print(f"{mask.sum():,} of {N_TOKENS:,} tokens pass the filters ({DEVICE}, {fit_cfg['epochs']:,} epochs)")

Now separate *when* from *how*: estimate the learning-time spectrum $p(\tau)$ of the fitted tokens, place one shared learning pulse $g$ at every learning time, and compare $(p * g)$ with the empirical loss derivative of **all** one million tokens.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from tokenlearning.core.rates import loss_change_rate
from tokenlearning.core.sigmoid import sigmoid_loss
from tokenlearning.core.spectrum import estimate_signed_spectrum, reconstruct_rate_spectrum_dominated

x_rate, empirical = loss_change_rate(steps, losses.mean(axis=0, dtype=np.float64))
recon = reconstruct_rate_spectrum_dominated(
    steps,
    fit,
    mask,
    dense_points=999,
    denominator=N_TOKENS,
    tau_range_for_g_mean=tuple(cfg["reconstruction"]["tau_range_for_g_mean"]),
)
spectrum = estimate_signed_spectrum(fit.tau[mask], fit.delta[mask], bins=cfg["spectrum"]["bins"])
r = np.corrcoef(empirical, np.interp(x_rate, recon.x_rate_dense, recon.reconstructed_rate_dense))[0, 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))

span = steps[-1] - steps[0]
inside = (fit.tau > steps[0] + 0.05 * span) & (fit.tau < steps[-1] - 0.05 * span)
sharp = 4.4 / fit.lambda_ < 0.1 * span  # 10-90% transition within 10% of training
candidates = np.flatnonzero(mask & inside & sharp & (fit.delta > 0.5) & (fit.start < 6))
examples = [group[np.argmax(fit.r2[group])] for group in np.array_split(candidates[np.argsort(fit.tau[candidates])], 4)]
colors = plt.cm.viridis(np.linspace(0.05, 0.85, len(examples)))
for idx, color in zip(examples, colors):
    curve = sigmoid_loss(steps, fit.tau[idx : idx + 1], fit.lambda_[idx : idx + 1], fit.start[idx : idx + 1], fit.end[idx : idx + 1])[0]
    axes[0].plot(steps, losses[rows[idx]], color=color, lw=1, alpha=0.55)
    axes[0].plot(steps, curve, color=color, lw=2, ls="--", label=rf"$\tau$ = {fit.tau[idx]:,.0f}")
axes[0].set(title="Tokens are learned in steps", xlabel="training step", ylabel="token loss")
axes[0].legend(frameon=False, fontsize=8)

keep = spectrum.pos_mass > 0
axes[1].loglog(spectrum.bin_centers[keep], spectrum.pos_mass[keep], "o-", color="#4CAF50", ms=3)
axes[1].set(title=r"Learning-time spectrum $p(\tau)$", xlabel=r"learning time $\tau$ (step)", ylabel="fraction of tokens")

axes[2].plot(x_rate, empirical, "o-", color="#1F4E79", ms=3, lw=1.2, label="Empirical loss derivative (all 1M tokens)")
axes[2].plot(recon.x_rate_dense, recon.reconstructed_rate_dense, "--", color="#C43C39", lw=2.2,
             label=f"Spectrum-dominated derivative ({N_TOKENS:,} tokens)")
axes[2].set(title=f"Loss derivative reconstruction (r = {r:.2f})", xlabel="training step", ylabel="loss derivative")
axes[2].legend(frameon=False, fontsize=8)

for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**What you should see.**

- **Left:** real tokens drop through localized transitions at very different training steps; dashed lines are their sigmoid fits.
- **Middle:** the learning-time spectrum is heavy-tailed. Many tokens are learned early and progressively fewer later, close to a straight line on log-log axes.
- **Right:** one shared pulse placed at the learning times of just 5% of the tokens reproduces the trend of the loss derivative of all one million. The empirical curve also fluctuates from checkpoint to checkpoint in the paper's Figure 3D.

The shape of the scaling curve is set by *when* tokens are learned.

## Next steps

- **Reproduce Figures 3–5**: `tokenlearning reproduce fig3 fig4 fig5 --quick` draws every panel from 50,000-token subsets in a few minutes on CPU; without `--quick` it runs the paper setting (GPU recommended). See the [README](https://github.com/applewpj/stepwise-token-learning#reproduce-the-paper).
- **Browse the data-scale and model-scale matrices** without fitting anything: the [Dataset Viewer](https://huggingface.co/datasets/applewpj/token-learning-spectrum-examples/viewer) shows 10,000-row previews of `d_axis_1m` and `m_axis_1m`.
- **Analyze your own checkpoints** with [`docs/own-data.md`](https://github.com/applewpj/stepwise-token-learning/blob/main/docs/own-data.md) and [`loss_acquisition/`](https://github.com/applewpj/stepwise-token-learning/tree/main/loss_acquisition).
- **Concepts and outputs** are explained in [`docs/concepts.md`](https://github.com/applewpj/stepwise-token-learning/blob/main/docs/concepts.md).

If this is useful in your work, please cite:

```bibtex
@inproceedings{wang2026smooth,
  title     = {Smooth Scaling Laws Hide Stepwise Token Learning},
  author    = {Wang, Pingjie and Hu, Zechen and Yang, Peiru and Guo, Fu and Zhang, Debing},
  booktitle = {Advances in Neural Information Processing Systems},
  year      = {2026},
  url       = {https://openreview.net/forum?id=THZDVZArRH}
}
```